# LongMemEval head-to-head: vstash vs ColBERTv2 (T4 Colab)

Same haystack, same chunking, same metric -- two retrieval engines.

| | vstash | ColBERTv2 |
|---|---|---|
| Model | BAAI/bge-small-en-v1.5 (33M params, 384d single-vec) | colbert-ir/colbertv2.0 (~110M, 128d per token) |
| Search | vector ANN + FTS5 + adaptive RRF + MMR | multi-vector MaxSim |
| Storage | sqlite-vec + FTS5 in one .db | per-question in-memory tensor |
| Per-q wall (T4) | ~0.5 s | ~3-5 s (encoding + MaxSim) |

Both run with the same `chunk_text(size=1024, overlap=128)`.
Both retrieve top-200 chunks then dedupe to unique sessions and
compute Recall@K against `answer_session_ids`.  ColBERT truncates
doc tokens at ~220, BGE-small at 512 -- documented as a caveat in
the result JSON.

Output: two parallel JSONs in `/content/results/`,
`lme_full_500_v3.json` (vstash hybrid) and
`lme_full_500_colbertv2.json` (ColBERTv2).  Cell 5 builds a side-
by-side table and saves it to Drive.

In [ ]:
# Cell 1: Setup -- vstash from the experiment branch + pylate for ColBERT.
BRANCH = 'feature/longmemeval-retrain-experiments'

!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0' pylate huggingface_hub
!rm -rf /content/vstash
!git clone --branch $BRANCH https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .

import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'
print('cuda:', torch.cuda.get_device_name(0),
      '| mem GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
# Cell 2: Download longmemeval_s.
import os
from huggingface_hub import hf_hub_download
TARGET = '/content/vstash/experiments/data/longmemeval'
os.makedirs(TARGET, exist_ok=True)
p = hf_hub_download('xiaowu0162/longmemeval', 'longmemeval_s',
                    repo_type='dataset', local_dir=TARGET)
print(f'Downloaded ({os.path.getsize(p) / 1024 / 1024:.1f} MB) -> {p}')

In [ ]:
# Cell 3: Run vstash retrieval (full 500 questions, hybrid mode).
# ~9 min with the daemon-cached embedder; matches the local Mac
# baseline number we already have at experiments/results/lme_full_500_v3.json.
import os
os.makedirs('/content/results', exist_ok=True)
os.chdir('/content/vstash')

!python -m experiments.longmemeval_retrieval \
    --all \
    --output /content/results/lme_full_500_vstash.json

In [ ]:
# Cell 4: Run ColBERTv2 retrieval (full 500 questions).
# Per-question fresh in-memory index; encoding dominates at ~3-5 s
# per question on T4.  Total ~30-45 min for 500 questions.
import os
os.chdir('/content/vstash')

!python -m experiments.longmemeval_colbert \
    --all \
    --device cuda \
    --encode-batch-size 32 \
    --output /content/results/lme_full_500_colbertv2.json

In [ ]:
# Cell 5: Side-by-side comparison + save to Drive (early, before any
# disconnect).  Produces both a printed table and a comparison JSON.
import json
import os
from collections import defaultdict

from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/lme_h2h'
os.makedirs(DRIVE_OUT, exist_ok=True)

v = json.load(open('/content/results/lme_full_500_vstash.json'))
c = json.load(open('/content/results/lme_full_500_colbertv2.json'))
ks = (1, 3, 5, 10, 20, 50)

rows = []
header = (
    f'{"K":>3} | {"vstash":>8} | {"ColBERT":>8} | {"delta":>8}'
)
rows.append(header)
rows.append('-' * len(header))
for k in ks:
    vk = v['summary']['macro'][f'recall@{k}']
    ck = c['summary']['macro'][f'recall@{k}']
    rows.append(f'{k:>3} | {vk:>8.4f} | {ck:>8.4f} | {vk - ck:+8.4f}')

rows.append('')
rows.append('Per question_type (Recall@10):')
for t in sorted(v['summary']['by_question_type']):
    vk = v['summary']['by_question_type'][t]['recall@10']
    ck = c['summary']['by_question_type'][t]['recall@10']
    nq = v['summary']['by_question_type'][t]['n']
    rows.append(
        f'  {t:30s} n={nq:3d} | vstash={vk:.4f} | ColBERT={ck:.4f} | delta={vk - ck:+.4f}'
    )
table = '\n'.join(rows)
print(table)

comparison = {
    'vstash': v['summary'],
    'colbertv2': c['summary'],
    'delta': {f'recall@{k}': v['summary']['macro'][f'recall@{k}']
              - c['summary']['macro'][f'recall@{k}'] for k in ks},
    'caveats': c['summary'].get('caveats', ''),
    'table': table,
}
with open('/content/results/lme_full_500_h2h.json', 'w') as fh:
    json.dump(comparison, fh, indent=2)

for src in ('lme_full_500_vstash.json', 'lme_full_500_colbertv2.json',
            'lme_full_500_h2h.json'):
    !cp /content/results/$src $DRIVE_OUT/$src
print(f'\nSaved to {DRIVE_OUT}/')